# PLN-THRML Quick Start

This notebook walks through the **complete 4-step pipeline** for compiling
PLN inference to thermodynamic factor graphs:

1. **Parameterize** — PLN `(stv s c)` → Beta(α, β)
2. **Build graph** — discretize into K bins, create nodes + factors
3. **Sample** — Block Gibbs sampling → raw categorical samples
4. **Recover** — posterior histogram → `(strength, confidence)`

Each step shows the intermediate data, so you can see exactly what happens
inside the pipeline.

In [ ]:
%pip install -e ..

In [ ]:
import jax.numpy as jnp

from pln_thrml.beta import (
    # Step 1: Parameterize
    stv_to_beta_params, c2w,
    # Step 2: Build graph
    DEFAULT_K, bin_centers, bin_width,
    beta_prior_weights, beta_implication_weights,
    make_beta_prior_factor, make_beta_implication_factor,
    build_beta_chain,
    # Step 3: Sample
    run_beta_sampling,
    DEFAULT_BETA_N_BATCHES, DEFAULT_BETA_SCHEDULE,
    # Step 4: Recover
    estimate_beta_marginal, posterior_to_stv,
    # Convenience wrapper (steps 3+4)
    sample_and_measure,
)

---
## Problem Setup: Modus Ponens

Given:
- P(A) = **(stv 0.8 0.9)** — "A is true with strength 0.8 and confidence 0.9"
- A → B with **(stv 0.9 0.85)**

Question: What is P(B)?

PLN analytical answer: **strength ≈ 0.724**

---
## Step 1: Parameterize — `(stv s c)` → Beta(α, β)

PLN truth values carry two numbers: **strength** (the estimate) and
**confidence** (how much evidence supports it). We convert each to a
Beta distribution: mean = strength, spread = confidence.

In [ ]:
# Node A: strong belief (stv 0.8 0.9)
alpha_A, beta_A = stv_to_beta_params(strength=0.8, confidence=0.9)
w_A = c2w(0.9)  # confidence → evidence weight
print(f"Node A: (stv 0.8 0.9)")
print(f"  evidence weight w = c/(1-c) = {w_A:.1f}")
print(f"  Beta(α={alpha_A:.2f}, β={beta_A:.2f})")
print(f"  mean = α/(α+β) = {alpha_A/(alpha_A+beta_A):.3f}  (= strength)")
print()

# Node B: weak prior (stv 0.5 0.01) — we know almost nothing
alpha_B, beta_B = stv_to_beta_params(strength=0.5, confidence=0.01)
w_B = c2w(0.01)
print(f"Node B: (stv 0.5 0.01)")
print(f"  evidence weight w = {w_B:.4f}  (nearly zero)")
print(f"  Beta(α={alpha_B:.4f}, β={beta_B:.4f})")
print(f"  mean = {alpha_B/(alpha_B+beta_B):.3f}  (= strength, but barely informative)")

---
## Step 2: Build Graph — discretize into K bins, create factors

Each Beta distribution is discretized into **K=16 bins** over [0, 1].
The log-probabilities become energy weights via the Boltzmann transform:
`P(x) ∝ e^{−ℰ(x)}`  →  `ℰ(x) = −log P(x)`

### 2a. Prior factors (unary, shape [K])

In [ ]:
K = DEFAULT_K
centers = bin_centers(K)
print(f"K={K} bins, centers: [{centers[0]:.3f}, {centers[1]:.3f}, ..., {centers[-1]:.3f}]")
print(f"Bin width: {bin_width(K):.4f}")
print()

# Prior weights for A: peaked around 0.8
w_prior_A = beta_prior_weights(0.8, 0.9, K)
print(f"Prior weights for A (first 5): {w_prior_A[:5]}")
peak_A = centers[jnp.argmin(w_prior_A)]
print(f"Peak bin for A: {peak_A:.3f}  (lowest energy = highest probability)")
print()

# Prior weights for B: nearly flat (uninformative)
w_prior_B = beta_prior_weights(0.5, 0.01, K)
print(f"Prior weights for B (first 5): {w_prior_B[:5]}")
print(f"Range: [{float(jnp.min(w_prior_B)):.3f}, {float(jnp.max(w_prior_B)):.3f}]  (nearly flat — no information)")

### 2b. Implication factor (pairwise, shape [K × K])

The implication A→B with (stv 0.9 0.85) becomes a K×K weight table.
Entry `[i, j]` is the log-probability of child bin `j` given parent bin `i`.

In [ ]:
w_impl = beta_implication_weights(0.9, 0.85, background=0.02, k=K)
print(f"Implication weight table shape: {w_impl.shape}  (parent_bins × child_bins)")
print()
print("Sample rows (parent bin → child distribution):")
for i in [0, 7, 15]:
    peak_j = centers[jnp.argmin(w_impl[i])]
    print(f"  parent={centers[i]:.3f} → child peaks at {peak_j:.3f}")

### 2c. Assemble the full graph

`build_beta_chain` combines priors + implications into a complete factor graph
with nodes, factors, and a Gibbs sampling program.

In [ ]:
graph = build_beta_chain(
    priors=[0.8, 0.5],
    confidences=[0.9, 0.01],
    strengths=[0.9],
    impl_confidences=[0.85],
    backgrounds=[0.02],
)

print(f"Graph keys: {list(graph.keys())}")
print(f"Nodes: {graph['n']} (K={graph['k']} bins each)")
print(f"Factors: {len(graph['factors'])} (priors + implications)")
print(f"Free blocks: {len(graph['free_blocks'])}  (nodes sampled by Gibbs)")
print(f"Clamped blocks: {len(graph['clamped_blocks'])}  (root clamped to prior)")

---
## Step 3: Sample — Block Gibbs sampling

The factor graph is sampled using Block Gibbs:
- **50 batches** × **2,000 samples** × **3 steps per sample**
- Root node A is **clamped** (sampled from its prior each batch)
- Free nodes (B) are updated by Gibbs sweeps

Output: raw categorical bin indices (integers 0..K-1).

In [ ]:
print(f"Sampling config: {DEFAULT_BETA_N_BATCHES} batches × "
      f"{DEFAULT_BETA_SCHEDULE.n_samples} samples × "
      f"{DEFAULT_BETA_SCHEDULE.steps_per_sample} steps/sample")
print()

samples = run_beta_sampling(graph, seed=42)

# samples is a list of arrays, one per free block
print(f"Number of free blocks: {len(samples)}")
print(f"Samples shape: {samples[0].shape}  (batches, samples, nodes_in_block)")
print(f"Sample dtype: {samples[0].dtype}  (categorical bin indices)")
print()

# Show raw samples: these are bin indices, not probabilities
print(f"First 10 samples from batch 0, node B:")
print(f"  bin indices: {samples[0][0, :10, 0]}")
print(f"  as strengths: {centers[samples[0][0, :10, 0]]}")

---
## Step 4: Recover — posterior → `(strength, confidence)`

Count how often each bin was visited → normalized histogram → moment-match
back to a Beta distribution → extract (strength, confidence).

In [ ]:
target = graph["nodes"][1]  # node B

# estimate_beta_marginal returns (posterior_histogram, strength, confidence)
posterior, strength, confidence = estimate_beta_marginal(samples, graph, target)

print(f"Posterior histogram (K={K} bins, sums to {float(jnp.sum(posterior)):.3f}):")
print(f"  {posterior}")
print()
print(f"Recovered truth value:")
print(f"  strength  = {strength:.4f}  (posterior mean)")
print(f"  confidence = {confidence:.4f}  (from posterior variance via moment-matching)")
print()
print(f"PLN analytical: strength ≈ 0.724")
print(f"Δ strength = {abs(strength - 0.724):.4f}")

### Alternative: `posterior_to_stv` on any histogram

You can also call `posterior_to_stv` directly on any K-bin histogram.

In [ ]:
s2, c2 = posterior_to_stv(posterior, K)
print(f"posterior_to_stv: (stv {s2:.4f} {c2:.4f})")
print(f"Same as above:   (stv {strength:.4f} {confidence:.4f})")

---
## Convenience Wrapper: `sample_and_measure`

In practice, steps 3+4 are combined into one call.
This is equivalent to what we did above.

In [ ]:
s, c = sample_and_measure(graph, target_node=graph["nodes"][1])
print(f"sample_and_measure: (stv {s:.4f} {c:.4f})")

---
## Deduction Chain: A → B → C

The same 4 steps work for longer chains. Here we measure the endpoint C
after two implications.

In [ ]:
graph3 = build_beta_chain(
    priors=[0.8, 0.5, 0.5],
    confidences=[0.9, 0.01, 0.01],
    strengths=[0.9, 0.85],
    impl_confidences=[0.85, 0.8],
    backgrounds=[0.02, 0.02],
)

s, c = sample_and_measure(graph3, target_node=graph3["nodes"][2])
print(f"P(C) = (stv {s:.4f} {c:.4f})")
print(f"Confidence drops along the chain — less evidence reaches C.")

---
## Next Steps

- **V-shape graphs** (induction): `build_beta_v_graph`
- **Inverted-V** (abduction): `build_beta_inv_v_graph`
- **Block-diagonal** partitioning for large graphs: `pln_thrml.block_diagonal`
- **MeTTa bridge**: `pln_thrml.metta` (requires `hyperon`)
- Run the full test suite: `pytest tests/ -v`